# MOD-08 · NB-04 — MLflow Model Tracking & Registry
### Health Analytics with Python · Module 08: Reproducible Research & Deployment
---
**Learning objectives**
- Understand MLflow's four core concepts: tracking, models, registry, projects
- Log parameters, metrics, and artifacts for every training run
- Compare multiple model runs in a structured experiment
- Register the best model and manage staging/production lifecycle
- Build a model metadata schema for clinical model governance
- Implement drift detection with MLflow metrics over time

**Estimated time:** 2.5 hours | **Level:** Intermediate-Advanced | **Libraries:** `mlflow`, `sklearn`

## 1. Setup & Dataset

In [ ]:
import os, json, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings; warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod08/mlruns", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 3000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,15,N).clip(18,90).astype(int)
los = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
dm  = np.random.binomial(1,0.35,N)
hf  = np.random.binomial(1,0.22,N)
copd= np.random.binomial(1,0.18,N)
logit=-3.2+0.6*hf+0.5*dm+0.3*copd+0.02*(age-60)+0.2*(los>7).astype(int)+np.random.normal(0,0.3,N)
readmitted = np.random.binomial(1,sigmoid(logit),N)
df = pd.DataFrame({"age":age,"los_days":los,"diabetes":dm,"hf":hf,"copd":copd,"readmitted":readmitted})
FEATURES = ["age","los_days","diabetes","hf","copd"]; TARGET = "readmitted"
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
print(f"Dataset: {N} patients | Readmission: {readmitted.mean()*100:.1f}%")
print(f"Train: {len(X_tr)} | Test: {len(X_te)}")


## 2. MLflow Architecture

```
MLflow Tracking Server
├── Experiments (logical groupings)
│   └── readmission_prediction/
│       ├── Run 001: LogisticRegression (AUC=0.72)
│       ├── Run 002: RandomForest       (AUC=0.76)
│       └── Run 003: GradientBoosting  (AUC=0.78) ← BEST
└── Artifacts
    └── models/, figures/, feature_importance.csv

MLflow Model Registry
├── readmission_model (registered model)
│   ├── Version 1: Staging  (AUC=0.72)
│   ├── Version 2: Archived (AUC=0.76)
│   └── Version 3: Production (AUC=0.78) ← SERVING
```


## 3. MLflow Tracking Setup

In [ ]:
try:
    import os
    os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
    import mlflow
    import mlflow.sklearn
    MLFLOW_OK = True
    print(f"MLflow {mlflow.__version__} available")
    mlflow.set_tracking_uri(f"file:///tmp/mod08/mlruns")
    mlflow.set_experiment("readmission_prediction")
except ImportError:
    MLFLOW_OK = False
    print("pip install mlflow  (proceeding with manual tracking fallback)")

# Manual MLflow-style tracking if MLflow not available
class SimpleTracker:
    def __init__(self, experiment_name, tracking_dir="/tmp/mod08/runs"):
        self.experiment = experiment_name
        self.runs = []
        self.tracking_dir = Path(tracking_dir)
        self.tracking_dir.mkdir(parents=True, exist_ok=True)

    def start_run(self, run_name):
        self.current_run = {"run_name":run_name,"params":{},"metrics":{},"tags":{},
                             "start_time":time.time()}
        return self

    def log_param(self, key, value):
        self.current_run["params"][key] = value

    def log_params(self, params):
        self.current_run["params"].update(params)

    def log_metric(self, key, value):
        self.current_run["metrics"][key] = round(float(value),6)

    def log_metrics(self, metrics):
        for k,v in metrics.items(): self.log_metric(k,v)

    def set_tag(self, key, value):
        self.current_run["tags"][key] = str(value)

    def end_run(self):
        self.current_run["duration_s"] = round(time.time()-self.current_run["start_time"],2)
        self.runs.append(dict(self.current_run))
        run_path = self.tracking_dir / f"{self.current_run['run_name']}.json"
        run_path.write_text(json.dumps(self.current_run, indent=2, default=str))

    def get_runs_df(self):
        if not self.runs: return pd.DataFrame()
        rows = []
        for r in self.runs:
            row = {"run_name":r["run_name"],"duration_s":r.get("duration_s",0)}
            row.update({f"param.{k}":v for k,v in r["params"].items()})
            row.update({f"metric.{k}":v for k,v in r["metrics"].items()})
            rows.append(row)
        return pd.DataFrame(rows)

tracker = SimpleTracker("readmission_prediction")
print("Tracking initialised.")
print("MLflow concepts this notebook covers:")
for concept in ["Experiments: group related runs","Runs: single model training",
                 "Parameters: hyperparameters logged","Metrics: AUC, AP, Brier",
                 "Artifacts: model files, figures, data","Tags: free-form metadata",
                 "Model Registry: versioned model store","UI: http://localhost:5000"]:
    print(f"  * {concept}")


## 4. Model Comparison Experiment

In [ ]:
# Run a model comparison experiment
model_configs = [
    {"name":"LogisticRegression_L2","model":LogisticRegression(C=1,max_iter=500,class_weight="balanced",random_state=42),
     "params":{"model_type":"LogisticRegression","C":1,"max_iter":500,"class_weight":"balanced"}},
    {"name":"LogisticRegression_strong","model":LogisticRegression(C=0.1,max_iter=500,class_weight="balanced",random_state=42),
     "params":{"model_type":"LogisticRegression","C":0.1,"max_iter":500,"class_weight":"balanced"}},
    {"name":"RandomForest","model":RandomForestClassifier(n_estimators=200,max_depth=6,class_weight="balanced",random_state=42),
     "params":{"model_type":"RandomForest","n_estimators":200,"max_depth":6,"class_weight":"balanced"}},
    {"name":"GradientBoosting","model":GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42),
     "params":{"model_type":"GradientBoosting","n_estimators":200,"max_depth":4,"learning_rate":0.05}},
]

print("Running model comparison experiment...")
print(f"{'Run Name':35s} {'AUC-ROC':>9s} {'Avg Prec':>10s} {'Brier':>8s} {'Time(s)':>9s}")
print("-"*75)

for config in model_configs:
    t0 = time.time()
    tracker.start_run(config["name"])
    tracker.log_params(config["params"])
    tracker.set_tag("dataset", "readmission_cohort")
    tracker.set_tag("features", str(FEATURES))

    config["model"].fit(X_tr, y_tr)
    prob = config["model"].predict_proba(X_te)[:,1]
    metrics = {
        "auc_roc"      : roc_auc_score(y_te, prob),
        "avg_precision": average_precision_score(y_te, prob),
        "brier_score"  : brier_score_loss(y_te, prob),
        "n_train"      : len(X_tr),
        "n_test"       : len(X_te),
        "positive_rate": float(y_te.mean()),
    }
    tracker.log_metrics(metrics)

    # Save model artifact
    model_dir = Path(f"/tmp/mod08/artifacts/{config['name']}")
    model_dir.mkdir(parents=True, exist_ok=True)
    pickle.dump(config["model"], open(model_dir/"model.pkl","wb"))

    elapsed = time.time()-t0
    tracker.end_run()
    print(f"  {config['name']:33s} {metrics['auc_roc']:>9.4f} {metrics['avg_precision']:>10.4f} "
          f"{metrics['brier_score']:>8.4f} {elapsed:>9.2f}")

runs_df = tracker.get_runs_df()
print("\nRun summary:")
display(runs_df[["run_name","metric.auc_roc","metric.avg_precision","metric.brier_score"]].round(4))


## 5. Model Registry & Lifecycle

In [ ]:
# Model Registry — promoting best model to staging/production
best_run = runs_df.loc[runs_df["metric.auc_roc"].idxmax(), "run_name"]
best_auc = runs_df["metric.auc_roc"].max()

print(f"Best model: {best_run} (AUC = {best_auc:.4f})")

# Model Registry concepts
REGISTRY_WORKFLOW = {
    "Staging"   : "Candidate model — integration tests passing",
    "Production": "Live model serving patient predictions",
    "Archived"  : "Previous production versions (for rollback)",
}

registry = {
    "model_name"     : "readmission_v1",
    "description"    : "30-day readmission prediction - LR baseline",
    "current_stage"  : "Production",
    "version_history": [
        {"version":1,"run_name":"LogisticRegression_L2","stage":"Archived",
         "auc":0.71,"promoted":"2023-06-01","note":"Initial baseline"},
        {"version":2,"run_name":"RandomForest","stage":"Archived",
         "auc":0.74,"promoted":"2023-09-01","note":"RF improved AUC by 3pp"},
        {"version":3,"run_name":best_run,"stage":"Production",
         "auc":best_auc,"promoted":"2024-01-15","note":"Current best model"},
    ]
}

print("\nModel Registry:")
print(f"  Name   : {registry['model_name']}")
print(f"  Stage  : {registry['current_stage']}")
print("\nVersion history:")
for v in registry["version_history"]:
    stage_icon = {"Production":"[PROD]","Archived":"[ARCH]"}.get(v["stage"],"[STAG]")
    print(f"  v{v['version']} {stage_icon} {v['run_name']:35s} AUC={v['auc']:.4f}  {v['note']}")

registry_path = Path("/tmp/mod08/model_registry.json")
registry_path.write_text(json.dumps(registry,indent=2,default=str))
print(f"\nRegistry saved: {registry_path}")


## 6. Experiment Visualisation

In [ ]:
# Compare experiment runs visually
metric_cols = [c for c in runs_df.columns if c.startswith("metric.") and c != "metric.n_train"]
run_names   = [r.split(".")[-1] for r in runs_df["run_name"]]

fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.suptitle("MLflow Experiment: Model Comparison", fontsize=13)
metrics_to_plot = ["metric.auc_roc","metric.avg_precision","metric.brier_score"]
metric_labels   = ["AUC-ROC","Average Precision","Brier Score"]
better          = ["higher","higher","lower"]
colors = ["#D65F5F","#4878CF","#6ACC65","#D4A017"]

for ax, metric, label, direction in zip(axes, metrics_to_plot, metric_labels, better):
    vals = runs_df[metric].values
    best_idx = vals.argmax() if direction=="higher" else vals.argmin()
    bar_colors = [colors[i] for i in range(len(runs_df))]
    bar_colors[best_idx] = "#1F4E79"  # highlight best
    bars = ax.bar(range(len(runs_df)), vals, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(runs_df)))
    ax.set_xticklabels([n[:10] for n in runs_df["run_name"]], rotation=20, fontsize=8)
    ax.set_ylabel(label)
    ax.set_title(f"{label}\n(best = {'max' if direction=='higher' else 'min'})")
    for i,(bar,val) in enumerate(zip(bars,vals)):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.002, f"{val:.4f}",
                ha="center", fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/mod08/mlflow_comparison.png",bbox_inches="tight")
plt.show()
print("Run comparison figure saved.")


## 7. MLflow CLI Commands

In [ ]:
# MLflow CLI and server commands
MLFLOW_COMMANDS = (
    "# Start MLflow tracking server (default: http://localhost:5000)\n"
    "mlflow server \\\n"
    "  --backend-store-uri sqlite:///mlflow.db \\\n"
    "  --default-artifact-root ./mlruns \\\n"
    "  --host 0.0.0.0 --port 5000\n\n"
    "# With S3 artifact storage (production):\n"
    "mlflow server \\\n"
    "  --backend-store-uri postgresql://user:pass@db:5432/mlflow \\\n"
    "  --default-artifact-root s3://your-bucket/mlflow-artifacts \\\n"
    "  --host 0.0.0.0 --port 5000\n\n"
    "# Python API tracking:\n"
    "import mlflow\n"
    "mlflow.set_tracking_uri('http://localhost:5000')\n"
    "with mlflow.start_run(run_name='GBM_v3'):\n"
    "    mlflow.log_params({'n_estimators':200,'lr':0.05})\n"
    "    mlflow.log_metric('auc', 0.789)\n"
    "    mlflow.sklearn.log_model(model, 'model',\n"
    "        registered_model_name='readmission_model')\n"
    "    mlflow.log_artifact('reports/figures/roc_curve.png')\n\n"
    "# Promote model to production:\n"
    "client = mlflow.MlflowClient()\n"
    "client.transition_model_version_stage(\n"
    "    name='readmission_model', version=3, stage='Production')\n\n"
    "# Load production model:\n"
    "model = mlflow.sklearn.load_model('models:/readmission_model/Production')\n"
    "predictions = model.predict_proba(X_new)[:,1]\n"
)
print(MLFLOW_COMMANDS)


## 8. Knowledge Check
1. You log 50 metrics per run. How do you decide which metrics are PRIMARY for model selection?
2. MLflow Model Registry has three stages: Staging, Production, Archived. What process should gate promotion from Staging to Production in a clinical context?
3. A new model has AUC=0.79 vs production model AUC=0.77. Should you always promote? What other factors matter?
4. Your MLflow database grows to 10GB after 6 months. What's your data management strategy?
5. Implement a model drift detector: log weekly AUC scores and alert when AUC drops >0.03 below the baseline.


In [ ]:
# Exercise 5 — Model drift detection
import matplotlib.pyplot as plt

# Simulate weekly AUC scores over 12 weeks (with drift in weeks 9-12)
np.random.seed(42)
weeks = range(1,13)
baseline_auc = 0.780
weekly_aucs = np.array([
    baseline_auc + np.random.normal(0,0.01)  # weeks 1-8: stable
    if w <= 8 else
    baseline_auc - 0.005*(w-8) + np.random.normal(0,0.01)  # weeks 9-12: drift
    for w in weeks
])
DRIFT_THRESHOLD = baseline_auc - 0.03

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(weeks, weekly_aucs, "o-", color="#4878CF", lw=2, ms=8, label="Weekly AUC")
ax.axhline(baseline_auc,     color="green",  ls="--", lw=1.5, label=f"Baseline AUC ({baseline_auc:.3f})")
ax.axhline(DRIFT_THRESHOLD,  color="#D65F5F",ls="--", lw=1.5, label=f"Alert threshold ({DRIFT_THRESHOLD:.3f})")
drift_weeks = [w for w,a in zip(weeks,weekly_aucs) if a < DRIFT_THRESHOLD]
for w in drift_weeks:
    ax.axvspan(w-0.4,w+0.4,alpha=0.15,color="#D65F5F")
    ax.plot(w,weekly_aucs[w-1],"v",color="#D65F5F",ms=12,zorder=5)
ax.set_xlabel("Week"); ax.set_ylabel("AUC-ROC")
ax.set_title("Model Performance Drift Monitoring\n(red markers = drift alert triggered)")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

print(f"Drift alerts triggered in weeks: {drift_weeks}")
print(f"Action required: investigate data pipeline and consider model retraining")


---
**Next → NB-05: FastAPI for Clinical Model Serving**